In [301]:
import numpy as np
import random
import time
from IPython.display import display, clear_output

# https://www.learndatasci.com/tutorials/reinforcement-q-learning-scratch-python-openai-gym/
# the Sutton textbook is the flipping best! 

In [294]:
"""
 +--------------+
 |00|01|02|03|04|
 |05|06|07|08|09|
 |10|11|12|13|14|
 |15|16|17|18|19|
 |20|21|22|23|24|
 +--------------+
"""


'\n\n +--------------+\n |00|01|02|03|04|\n |05|06|07|08|09|\n |10|11|12|13|14|\n |15|16|17|18|19|\n |20|21|22|23|24|\n +--------------+\n \n'

In [992]:
def get_row_col_nums(square_number):
    if square_number in range(0,5):
        row_num = 0
    elif square_number in range(5,10):
        row_num = 1
    elif square_number in range(10,15):
        row_num = 2
    elif square_number in range(15,20):
        row_num = 3
    elif square_number in range(20,25):
        row_num = 4
    else:
        row_num = None
        
    if square_number in range(0,21,5):
        col_num = 0
    elif square_number in range(1,22,5):
        col_num = 1
    elif square_number in range(2,23,5):
        col_num = 2
    elif square_number in range(3,24,5):
        col_num = 3
    elif square_number in range(4,25,5):
        col_num = 4
    else:
        row_num = None    

    return row_num, col_num    
        
for i in range(0,25):
    print( f'square {i} = row {get_row_col_nums(i)[0]}, col {get_row_col_nums(i)[1]}' )

square 0 = row 0, col 0
square 1 = row 0, col 1
square 2 = row 0, col 2
square 3 = row 0, col 3
square 4 = row 0, col 4
square 5 = row 1, col 0
square 6 = row 1, col 1
square 7 = row 1, col 2
square 8 = row 1, col 3
square 9 = row 1, col 4
square 10 = row 2, col 0
square 11 = row 2, col 1
square 12 = row 2, col 2
square 13 = row 2, col 3
square 14 = row 2, col 4
square 15 = row 3, col 0
square 16 = row 3, col 1
square 17 = row 3, col 2
square 18 = row 3, col 3
square 19 = row 3, col 4
square 20 = row 4, col 0
square 21 = row 4, col 1
square 22 = row 4, col 2
square 23 = row 4, col 3
square 24 = row 4, col 4


In [1175]:
def calc_new_pos(current_coords, action):
    # note: current coords is a tuple with format (rownum,colnum)
    
    current_row = current_coords[0]
    current_col = current_coords[1]
    new_row = current_row
    new_col = current_col
    
    if (action == 'left') & (current_col > 0): # i.e. if there is space to go left
        new_col = current_col -1
    elif (action == 'up') & (current_row > 0):  # i.e. if there is space to go up
        new_row = current_row -1
    elif (action == 'right') & (current_col < 4):   # i.e. if there is space to go right
        new_col = current_col +1
    elif (action == 'down') & (current_row < 4):    # i.e. if there is space to go right
        new_row = current_row +1

    return (new_row, new_col)

In [1051]:
"""
   0 1 2 3 4
  +---------+
0 |_|_|_|_|_|
1 |_|_|_|_|_|
2 |_|_|_|_|_|
3 |_|_|_|_|_|
4 |_|_|_|_|_|
  +---------+
"""

calc_new_pos( current_coords = (2,2), 
              action = 'right'
            )

(2, 3)

In [1053]:
# code states as agent_row, agent_col, enemy_row, enemy_col
# e.g. 2_1__3_0
# dictionary keys are coded: AgentPosition_EnemyPosition    ( e.g. 4_0 )
state_ref = dict( zip([ str(a) + '_' + str(b) + '__' + str(c) + '_' + str(d) for a in range(5) for b in range(5) for c in range(5) for d in range (5) ], range(25*25)) )
state_ref

{'0_0__0_0': 0,
 '0_0__0_1': 1,
 '0_0__0_2': 2,
 '0_0__0_3': 3,
 '0_0__0_4': 4,
 '0_0__1_0': 5,
 '0_0__1_1': 6,
 '0_0__1_2': 7,
 '0_0__1_3': 8,
 '0_0__1_4': 9,
 '0_0__2_0': 10,
 '0_0__2_1': 11,
 '0_0__2_2': 12,
 '0_0__2_3': 13,
 '0_0__2_4': 14,
 '0_0__3_0': 15,
 '0_0__3_1': 16,
 '0_0__3_2': 17,
 '0_0__3_3': 18,
 '0_0__3_4': 19,
 '0_0__4_0': 20,
 '0_0__4_1': 21,
 '0_0__4_2': 22,
 '0_0__4_3': 23,
 '0_0__4_4': 24,
 '0_1__0_0': 25,
 '0_1__0_1': 26,
 '0_1__0_2': 27,
 '0_1__0_3': 28,
 '0_1__0_4': 29,
 '0_1__1_0': 30,
 '0_1__1_1': 31,
 '0_1__1_2': 32,
 '0_1__1_3': 33,
 '0_1__1_4': 34,
 '0_1__2_0': 35,
 '0_1__2_1': 36,
 '0_1__2_2': 37,
 '0_1__2_3': 38,
 '0_1__2_4': 39,
 '0_1__3_0': 40,
 '0_1__3_1': 41,
 '0_1__3_2': 42,
 '0_1__3_3': 43,
 '0_1__3_4': 44,
 '0_1__4_0': 45,
 '0_1__4_1': 46,
 '0_1__4_2': 47,
 '0_1__4_3': 48,
 '0_1__4_4': 49,
 '0_2__0_0': 50,
 '0_2__0_1': 51,
 '0_2__0_2': 52,
 '0_2__0_3': 53,
 '0_2__0_4': 54,
 '0_2__1_0': 55,
 '0_2__1_1': 56,
 '0_2__1_2': 57,
 '0_2__1_3': 58,
 '0_2__

In [1058]:
# create the Q-table, and fill it with zeros:
# q table is (n states) x (n actions)
# actions are ['left','up','right','down','stay']
q_table = np.zeros([25*25, 5])
q_table

array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       ...,
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

In [1059]:
# define some constants:

# this is the reward that the agent receives for every step that it remains alive:
ALIVE_REWARD = 1

# define the penalty given to the agent when it is killed by the enemy:
DEATH_REWARD = -50

In [1208]:
# define the environment:
class env_class:
    def __init__(self):
        self.agent_position = [4,4]
        self.enemy_position = [0,0]
        self.action_space = ['left','up','right','down','stay']
        self.is_game_over = False

    def render(self):   # print current state of environment  
        grid_to_print = [[' ' for x in range(5)] for y in range(5)]   # create a grid of blank characters:
        grid_to_print[self.agent_position[0]][self.agent_position[1]] = '*'   # add in the agent
        grid_to_print[self.enemy_position[0]][self.enemy_position[1]] = 'X'   # add in the enemy
        
        # print the game board:
        print( '  0 1 2 3 4' )
        print( ' +---------+' )
        for row_i in range(5):
            print( str(row_i) + '|' + '|'.join(grid_to_print[row_i]) + '|' )
        print( ' +---------+' )    
            
    # function to reset the agent and enemy to new random positions:
    # (i.e. restart the game)
    def reset(self):    
        self.agent_position = [ random.randint(0,4), random.randint(0,4) ] 
        self.enemy_position = [ random.randint(0,4), random.randint(0,4) ]

        # ensure that enemy and agent can't start on the same square:
        while self.agent_position == self.enemy_position:
            self.enemy_position = [ random.randint(0,4), random.randint(0,4) ]
        
        self.is_game_over = False     
        
    # define next state of the system after agent chooses an action:    
    def step(self, agent_action):
        
        # calculate new distance between enemy & agent under each possible enemy action
        # (enemy chooses to move in a way that gets it closest to the agent, assuming the agent won't move)
        enemy_dist_values = [0,0,0,0]  # left, up, right, down    (note: enemy won't stand still)

        new_enemy_coords_if_go_left = calc_new_pos(current_coords=self.enemy_position, action='left')
        new_enemy_coords_if_go_up = calc_new_pos(current_coords=self.enemy_position, action='up')
        new_enemy_coords_if_go_right = calc_new_pos(current_coords=self.enemy_position, action='right')
        new_enemy_coords_if_go_down = calc_new_pos(current_coords=self.enemy_position, action='down')
        
        # distance to agent if go left:
        enemy_dist_values[0] = sum( np.abs( np.subtract(new_enemy_coords_if_go_left, self.agent_position) ) )
        # distance to agent if go up:
        enemy_dist_values[1] = sum( np.abs( np.subtract(new_enemy_coords_if_go_up, self.agent_position) ) )
        # distance to agent if go right:
        enemy_dist_values[2] = sum( np.abs( np.subtract(new_enemy_coords_if_go_right, self.agent_position) ) )
        # distance to agent if go down:
        enemy_dist_values[3] = sum( np.abs( np.subtract(new_enemy_coords_if_go_down, self.agent_position) ) )

        # choose action getting closest to agent:
        chosen_enemy_action = ['left','up','right','down'][ np.argmin(enemy_dist_values) ]        
        
        # update position of enemy:
        self.enemy_position = calc_new_pos( current_coords = self.enemy_position, 
                                                    action = chosen_enemy_action
                                          )
        
        # update position of agent:
        self.agent_position = calc_new_pos( current_coords = self.agent_position, 
                                                    action = agent_action
                                          )
                
        if self.enemy_position == self.agent_position:
            self.is_game_over = True
            reward = DEATH_REWARD
        else:
            is_game_over = False
            reward = ALIVE_REWARD   
        
        return reward
        
env = env_class()

In [1209]:
# set enviroment to a random starting state:
env.reset()
print( f'agent position: {env.agent_position}' )
print( f'enemy position: {env.enemy_position}' )

env.render()

agent position: [0, 4]
enemy position: [1, 0]
  0 1 2 3 4
 +---------+
0| | | | |*|
1|X| | | | |
2| | | | | |
3| | | | | |
4| | | | | |
 +---------+


# Epsilon-Greedy Double Q-Learning Algorithm

Keep 2 tables $Q_1$ and $Q_2$, both containing estimates of the value of each action $A$ under each possible state $S$ 

Model hyperparameters (you must choose these):

1. $\epsilon$ (exploration rate)

2. $\gamma$ (discount rate)

3. $\alpha$ (learning rate)

Perform the following steps repeatedly:

1. Choose a random value $u$ uniformly from $[0,1]$

2. If $u<\epsilon$:

      .......then EXPLORE (choose a random action A)
        
   else if $u\geq epsilon$:
        
      .......then EXPLOIT (choose action a with highest estimated value, where value of each action A is estimated as $Q_1(A) + Q_2(A)$



In [1211]:
past_survival_times = []      # to store survival time in each game/episode (can also see how many games played)
survival_turn_counter = 0     # to count how many steps the agent has survived in each game/episode

for i in range(100):

    print(f'steps survived this game/episode: {survival_turn_counter}')
    
    # if game is over, then start the game again, 
    if env.is_game_over == True:
        env.reset()
        past_survival_times.append( survival_turn_counter )
        survival_turn_counter = 0
    
    agent_action = random.sample( env.action_space, 1 )[0]
    
    reward = env.step(agent_action)    
    
    env.render()
        
    print(reward)
    print(env.is_game_over)
    
    if env.is_game_over == True:
        print('GAME OVER')
    else:
        survival_turn_counter += 1
        
    clear_output(wait=True)
    time.sleep(0.5)

steps survived this game/episode: 2
  0 1 2 3 4
 +---------+
0| |*| | | |
1| | | | | |
2| | | | | |
3| |X| | | |
4| | | | | |
 +---------+
1
False


In [1164]:
random.sample( env.action_space, 1 )[0]

'left'

'down'

In [249]:
# the current state of the environment is stored in "s":
env.s

0

In [250]:
# the space of possible actions is stored in "action_space":
env.action_space

['up', 'down', 'stay']

In [251]:
# taking an action updates the state:
for action_i in ['down','down','up','down','stay','down','down','stay','up']:
    print(f'{[action_i]}')
    env.step(action_i)
    print(f'state: {env.s}\n')
    env.render()
    clear_output(wait=True)
    time.sleep(1)

['up']
state: 3

00
01
02
03 *
04
05
06
07




In [252]:
# env.reward_from_action() gives the reward for a given action, 
# given the state that we're currently in
print( f'current state: {env.s} ' )
print( f'reward if move up: {env.reward_from_action("up")}' )
print( f'reward if stay: {env.reward_from_action("stay")}' )
print( f'reward if move down: {env.reward_from_action("down")}' )

current state: 3 
reward if move up: 2
reward if stay: -1
reward if move down: -2


In [253]:
# the state can be reset back to the starting state again using env.reset():
env.reset()
env.render()

00 *
01
02
03
04
05
06
07




In [279]:
q_table = np.zeros([8, 3])
q_table

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [280]:
# Hyperparameters
alpha = 0.3             # 
gamma = 0.8             # discount rate
epsilon = 1           # probability of exploration

# length of each game (episode)
episode_length = 100

In [283]:
env.reset()
step_counter = 0    # to keep track of when each game (episode) has ended
episode_counter = 1

for i in range(1000):
    print(f'episode {episode_counter}')
    print(f'step {step_counter}\n')
    print(f'epsilon = {epsilon}')
    
    # reduce epsilon:
    epsilon = max( 0.05, epsilon*0.9999 )
    
    # choose an action:    
    if random.uniform(0, 1) < epsilon:
        print('explore')
        action = random.sample( env.action_space, 1 )[0]  # explore
    else:
        print('exploit')
        action = env.action_space[ np.argmax( q_table[env.s] ) ]  # exploit
    
    print(f'[{action}]')
    
    print(f'state {env.s}')
    env.render()    
        
    # get reward for action:
    reward = env.reward_from_action( action )    
    
    action_index = np.where( [ x==action for x in env.action_space ] )[0][0]
    print( action_index )
    state = env.s
    old_value = q_table[state, action_index]
    
    # update state:
    env.step( action )
    next_max = np.max( q_table[env.s] )     # max for next state 
    
    # update Q-table:
    new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
    q_table[state, action_index] = new_value
    print( env.action_space )
    print( q_table )                  
    
    # iterate step counter:
    step_counter += 1
    
    if step_counter >= episode_length:
        env.reset()
        episode_counter += 1
        step_counter = 0
    else:    
        step_counter += 1    
    
    clear_output(wait=True)
    time.sleep(0.05)
    

KeyboardInterrupt: 